# Train LFW Many People Few Images

Representative role: `many_people_few_images`.

This notebook processes LFW with the `100 x 10` preset, splits train/test before saving, then trains `PCA + KNN` and `PCA + SVM` from the persisted split.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if not (ROOT / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT


In [ ]:
import pandas as pd

from src.process import process_lfw_many_people_few_images_dataset
from src.pipelines import train_pca_knn, train_pca_svm
from src.utils import compare_models, plot_confusion_matrix, plot_explained_variance, save_metrics


## Training config

In [ ]:
config = {
    "dataset_name": "lfw",
    "role": "many_people_few_images",
    "test_size": 0.2,
    "random_state": 42,
    "n_components": 20,
    "knn_k": 3,
    "knn_metric": "euclidean",
    "svm_C": 1.0,
    "svm_kernel": "linear",
    "svm_gamma": "scale",
    "svm_max_iter": 50,
}

pd.Series(config)


## Process dataset, split train/test, and save processed artifacts

In [ ]:
processed = process_lfw_many_people_few_images_dataset(
    test_size=config["test_size"],
    random_state=config["random_state"],
    save_artifacts=True,
)

summary = processed["summary"]
label_names = processed["metadata"]["label_names"]
confusion_labels = label_names if len(label_names) <= 20 else None

X_train = processed["X_train"]
X_test = processed["X_test"]
y_train = processed["y_train"]
y_test = processed["y_test"]

pd.Series({
    "dataset_name": summary["dataset_name"],
    "role": config["role"],
    "samples_total": summary["samples_total"],
    "classes_total": summary["classes_total"],
    "train_shape": summary["train_shape"],
    "test_shape": summary["test_shape"],
    "stratify_used": summary["stratify_used"],
    "processed_dir": processed["output_dir"],
})


## Train PCA + KNN

In [ ]:
knn_model = train_pca_knn(
    X_train,
    y_train,
    n_components=config["n_components"],
    k=config["knn_k"],
    metric=config["knn_metric"],
)

knn_eval = knn_model.evaluate(X_test, y_test)
pd.Series({
    "accuracy": knn_eval["accuracy"],
    "train_time": knn_eval["train_time"],
})


In [ ]:
plot_explained_variance(knn_model.pca.explained_variance_ratio_)


In [ ]:
plot_confusion_matrix(knn_eval["confusion_matrix"], labels=confusion_labels)


## Train PCA + SVM

In [ ]:
svm_model = train_pca_svm(
    X_train,
    y_train,
    n_components=config["n_components"],
    C=config["svm_C"],
    kernel=config["svm_kernel"],
    gamma=config["svm_gamma"],
    max_iter=config["svm_max_iter"],
)

svm_eval = svm_model.evaluate(X_test, y_test)
pd.Series({
    "accuracy": svm_eval["accuracy"],
    "train_time": svm_eval["train_time"],
})


In [ ]:
plot_explained_variance(svm_model.pca.explained_variance_ratio_)


In [ ]:
plot_confusion_matrix(svm_eval["confusion_matrix"], labels=confusion_labels)


## Compare and save metrics

In [ ]:
comparison_df = compare_models(
    {
        "PCA+KNN": knn_eval,
        "PCA+SVM": svm_eval,
    }
)

comparison_df


In [ ]:
metrics_path = ROOT / "results" / "metrics" / "lfw_many_people_few_images_comparison.csv"
save_metrics(comparison_df, metrics_path)
metrics_path


## Optional: save trained models

In [ ]:
model_dir = ROOT / "webapp" / "saved_models"
model_dir.mkdir(parents=True, exist_ok=True)

knn_path = model_dir / "pca_knn_lfw_many_people_few_images.pkl"
svm_path = model_dir / "pca_svm_lfw_many_people_few_images.pkl"

knn_model.save_model(knn_path)
svm_model.save_model(svm_path)

pd.Series({
    "knn_model": str(knn_path),
    "svm_model": str(svm_path),
})
